# B3: global QoIs for the local-refinement checkpoint + J=det(F) validity check

Two real gaps flagged in the paper review, neither needing a new training run:

1. The local-integration-refinement checkpoint (`b3_training_local_refine/checkpoint_50000.pt`) improved regional Cauchy-stress error from ~62% to ~29.8%, but its own displacement/energy/reaction were never separately measured -- the paper currently reports those three only for the baseline checkpoint (`b3_training_normalized/checkpoint_50000.pt`).
2. No deformation-validity (J = det F > 0) check has ever been run for B3. The constitutive energies use ln(J), so this matters.

This notebook runs `evaluate_B3_global_qois_and_jacobian.py` on **both** checkpoints (baseline, then local-refinement) and prints both results side by side at the end for easy copy-paste back.

In [ ]:
import os, subprocess, sys

def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'timm', 'h5py', 'jax', 'tqdm'])
# Real, currently-active PyPI issue (per this project's established fix,
# e.g. cell_b8_gpu_mesh_convergence.py): pyvista 0.49+ unconditionally
# imports IPython.core.guarded_eval, which only exists from IPython>=8.8 --
# Colab ships an older IPython, so a fresh `pip install torch-fem` (which
# pulls in whatever pyvista is "latest", unpinned) can fail with
# ModuleNotFoundError. Pin pyvista below 0.49 BEFORE installing torch-fem.
run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')):
        del sys.modules[_mod_name]

print('Ready. Working dir:', os.getcwd())

## Run 1: baseline checkpoint (`b3_training_normalized`)

This is the checkpoint the paper's current displacement/energy/reaction/break-even numbers (1.88%/0.30%/2.67%) already come from -- running it here too gives the J-diagnostic for the paper's primary checkpoint, and confirms this script reproduces the already-known displacement number as a sanity check.

In [ ]:
BASE = '/content/drive/MyDrive/pfem_run'

run([
    'python', '-u', '-m', 'omar_pfem.evaluate_B3_global_qois_and_jacobian',
    '--checkpoint', f'{BASE}/b3_training_normalized/checkpoint_50000.pt',
    '--dataset', f'{BASE}/b3_dataset_clean_holdout/dataset.h5',
    '--out_json', f'{BASE}/b3_training_normalized/global_qois_and_jacobian.json',
])

## Run 2: local-integration-refinement checkpoint (`b3_training_local_refine`)

This is the NEW measurement the paper is missing: does the checkpoint that halved the regional-stress error still reproduce displacement/energy/reaction well?

In [ ]:
run([
    'python', '-u', '-m', 'omar_pfem.evaluate_B3_global_qois_and_jacobian',
    '--checkpoint', f'{BASE}/b3_training_local_refine/checkpoint_50000.pt',
    '--dataset', f'{BASE}/b3_dataset_clean_holdout/dataset.h5',
    '--out_json', f'{BASE}/b3_training_local_refine/global_qois_and_jacobian.json',
])

## Side-by-side summary (copy this cell's output back)

In [ ]:
import json

with open(f'{BASE}/b3_training_normalized/global_qois_and_jacobian.json') as f:
    baseline = json.load(f)
with open(f'{BASE}/b3_training_local_refine/global_qois_and_jacobian.json') as f:
    local_refine = json.load(f)

print('=== BASELINE (b3_training_normalized) ===')
print(json.dumps(baseline, indent=2))
print()
print('=== LOCAL-REFINEMENT (b3_training_local_refine) ===')
print(json.dumps(local_refine, indent=2))